In [64]:

"""Part 3 - Machine Learning: predict no-shows (Tasks 3.1 - 3.5)."""
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     cross_val_score, cross_val_predict)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)

UP = "/ANGAD/DOWNLOAD/uploads/"
CLEAN = "appointments_clean.csv"


In [55]:

# ---------------------------------------------------------------------------
# PART 3 — LOAD PART 1'S CLEANED DATASET
# If appointments_clean.csv is missing, rebuild it from appointment.csv
# ---------------------------------------------------------------------------

import os
import pandas as pd
import numpy as np

# File paths
UP = "./"   # Change this if your CSV files are in another folder
CLEAN = UP + "appointments_clean.csv"


# ---------------------------------------------------------------------------
# STEP 1 — REBUILD CLEANED DATASET IF REQUIRED
# ---------------------------------------------------------------------------

def rebuild_clean():

    # Load original appointment dataset
    d = pd.read_csv(UP + "appointment.csv")

    # Remove duplicate records (1020 -> 1000)
    d = d.drop_duplicates()

    # Remove rows with missing target values
    d = d.dropna(subset=["status"])

    # Standardize booking channel
    d["booking_channel"] = (
        d["booking_channel"]
        .str.strip()
        .str.lower()
        .map({
            "app": "App",
            "phone": "Phone",
            "walk-in": "Walk-in"
        })
    )

    # Convert date columns into datetime format
    d["booked_on"] = pd.to_datetime(
        d["booked_on"], errors="coerce"
    )

    d["appt_date"] = pd.to_datetime(
        d["appt_date"], errors="coerce"
    )

    # Remove invalid or missing dates
    d = d.dropna(subset=["booked_on", "appt_date"])

    # Calculate lead days
    d["lead_days"] = (
        d["appt_date"] - d["booked_on"]
    ).dt.days

    # Replace impossible ages with missing values
    d.loc[d["age"] <= 0, "age"] = np.nan

    # Handle outliers using IQR and fill missing values with median
    for c in ["age", "distance_km", "fee_inr"]:

        q1, q3 = d[c].quantile([0.25, 0.75])

        iqr = q3 - q1

        lo = q1 - 1.5 * iqr
        hi = q3 + 1.5 * iqr

        # Replace outliers with NaN
        d.loc[(d[c] < lo) | (d[c] > hi), c] = np.nan

        # Fill missing values with median
        d[c] = d[c].fillna(d[c].median())

    # Create target variable
    # No-Show = 1, Attended = 0
    d["no_show"] = (
        d["status"] == "No-Show"
    ).astype(int)

    # Reset index without changing row order
    return d.reset_index(drop=True)


# ---------------------------------------------------------------------------
# STEP 2 — CHECK AND LOAD CLEANED FILE
# ---------------------------------------------------------------------------

if os.path.exists(CLEAN):

    print("Existing appointments_clean.csv found.")
    df = pd.read_csv(CLEAN)

else:

    print("appointments_clean.csv not found.")
    print("Rebuilding from original appointment.csv...")

    df = rebuild_clean()

    # Save cleaned dataset
    df.to_csv(CLEAN, index=False)

    print("Cleaned dataset successfully created!")


# ---------------------------------------------------------------------------
# STEP 3 — DISPLAY DATASET INFORMATION
# ---------------------------------------------------------------------------

print("\n========== DATASET INFORMATION ==========")

print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nNo-show distribution:")
print(df["no_show"].value_counts())

print("\nNo-show percentage:")
print(df["no_show"].value_counts(normalize=True) * 100)

print("\nFinal cleaned dataset loaded successfully!")


appointments_clean.csv not found.
Rebuilding from original appointment.csv...
Cleaned dataset successfully created!

========== DATASET INFORMATION ==========
Dataset shape: (990, 16)

First 5 rows:
  appointment_id patient_id clinic_id  booked_on  appt_date   age gender  \
0          A0001       P509       C05 2025-01-24 2025-01-25  60.0      F   
1          A0002       P046       C07 2025-05-27 2025-06-06  39.0      M   
2          A0003       P009       C02 2025-03-24 2025-03-30  63.0      F   
3          A0004       P482       C01 2025-04-18 2025-04-22  52.0      F   
4          A0005       P251       C01 2024-11-23 2025-01-02  64.0      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0    850.0           Phone  Consultation           Yes   
1         12.5    350.0         Walk-in     Follow-up            No   
2          4.1    850.0         Walk-in  Consultation            No   
3          7.5    750.0             App  Consultation       

In [56]:
# ===========================================================================
# Task 3.1 - Features, split, baseline
# ===========================================================================
print("\n=== Task 3.1 ===")
ids = df["appointment_id"]                               # keep aside
y = df["no_show"]
X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
        "reminder_sent", "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
print("X columns:", X.shape[1], list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("No-shows in test set:", int(y_test.sum()), "| overall no-show rate: %.1f%%" % (100 * y.mean()))
base = (y_test == 0).mean()
print("Baseline accuracy (always 'Attended'): %.4f (%d of %d)" % (base, (y_test == 0).sum(), len(y_test)))



=== Task 3.1 ===
X columns: 11 ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-in', 'visit_type_Consultation', 'visit_type_Follow-up']
Train rows: 792 | Test rows: 198
No-shows in test set: 49 | overall no-show rate: 22.0%
Baseline accuracy (always 'Attended'): 0.7525 (149 of 198)


In [57]:
# ===========================================================================
# Task 3.2 - Logistic regression
# ===========================================================================
print("\n=== Task 3.2 ===")
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)                # fit on training only
X_test_s = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
pred = lr.predict(X_test_s)
prob = lr.predict_proba(X_test_s)[:, 1]
lr_acc, lr_rec, lr_auc = accuracy_score(y_test, pred), recall_score(y_test, pred), roc_auc_score(y_test, prob)
print("Accuracy : %.3f" % lr_acc)
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("Precision: %.3f  Recall: %.3f  F1: %.3f  ROC-AUC: %.3f" %
      (precision_score(y_test, pred), lr_rec, f1_score(y_test, pred), lr_auc))
coefs = pd.Series(lr.coef_[0], index=X.columns)
print("\nCoefficients (scaled features):\n", coefs.round(3).to_string())
i = list(X.columns).index("reminder_sent")
c_scaled = lr.coef_[0][i]
c_raw = c_scaled / scaler.scale_[i]
print("\nreminder_sent SD = %.3f" % scaler.scale_[i])
print("exp(coef) per 1 SD (scaled)  : %.3f" % np.exp(c_scaled))
print("exp(coef/SD) for No -> Yes   : %.3f  (odds ~%.0f%% lower)" % (np.exp(c_raw), 100 * (1 - np.exp(c_raw))))
print("Why they differ: the first is the odds ratio for a one-standard-deviation move in the scaled "
      "feature (SD ~0.49), whereas the second rescales to the real 0 -> 1 step (a full unit, ~2 SDs).")
print("Conclusion: in this model, holding the other features constant, appointments with a reminder have "
      "about 69% lower odds of a no-show - an association in observational data, not proof that reminders cause it.")


=== Task 3.2 ===
Accuracy : 0.783
Confusion matrix:
 [[140   9]
 [ 34  15]]
Precision: 0.625  Recall: 0.306  F1: 0.411  ROC-AUC: 0.781

Coefficients (scaled features):
 age                       -0.250
distance_km                0.199
fee_inr                    0.467
lead_days                  0.614
prior_no_shows             0.652
reminder_sent             -0.587
booking_channel_App       -0.045
booking_channel_Phone      0.219
booking_channel_Walk-in   -0.192
visit_type_Consultation    0.010
visit_type_Follow-up      -0.010

reminder_sent SD = 0.495
exp(coef) per 1 SD (scaled)  : 0.556
exp(coef/SD) for No -> Yes   : 0.306  (odds ~69% lower)
Why they differ: the first is the odds ratio for a one-standard-deviation move in the scaled feature (SD ~0.49), whereas the second rescales to the real 0 -> 1 step (a full unit, ~2 SDs).
Conclusion: in this model, holding the other features constant, appointments with a reminder have about 69% lower odds of a no-show - an association in observat

In [58]:

# Task 3.3 - Decision tree and overfitting
# ===========================================================================
print("\n=== Task 3.3 ===")
tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
for name, t in [("Full-depth tree", tree_full), ("Depth-3 tree", tree3)]:
    print("%-16s train acc %.3f | test acc %.3f%s" % (
        name, t.score(X_train, y_train), t.score(X_test, y_test),
        " (depth %d)" % t.get_depth() if t is tree_full else ""))
print("Gap: the full tree scores 1.00 on training but falls below the %.4f baseline on test, i.e. it "
      "memorised noise; the depth-3 tree has a small gap and generalises." % base)


=== Task 3.3 ===
Full-depth tree  train acc 1.000 | test acc 0.641 (depth 14)
Depth-3 tree     train acc 0.819 | test acc 0.768
Gap: the full tree scores 1.00 on training but falls below the 0.7525 baseline on test, i.e. it memorised noise; the depth-3 tree has a small gap and generalises.


In [60]:

# Task 3.4 - Random forest and comparison table #

print("\ Task 3.4 ")
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]
print("RF train acc %.3f | test acc %.3f | recall %.3f | ROC-AUC %.3f" % (
    rf.score(X_train, y_train), accuracy_score(y_test, rf_pred),
    recall_score(y_test, rf_pred), roc_auc_score(y_test, rf_prob)))
print("Top 3 importances:\n", pd.Series(rf.feature_importances_, index=X.columns)
      .sort_values(ascending=False).head(3).round(3).to_string())

t3_pred = tree3.predict(X_test)
t3_prob = tree3.predict_proba(X_test)[:, 1]
table = pd.DataFrame({
    "Model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "Accuracy": [lr_acc, accuracy_score(y_test, t3_pred), accuracy_score(y_test, rf_pred)],
    "Recall": [lr_rec, recall_score(y_test, t3_pred), recall_score(y_test, rf_pred)],
    "ROC-AUC": [lr_auc, roc_auc_score(y_test, t3_prob), roc_auc_score(y_test, rf_prob)],
}).round(3)
print("\n", table.to_string(index=False))
best_row = table.loc[table["ROC-AUC"].idxmax(), "Model"]
print("\nOn this (reporting-only) table, %s looks best: highest ROC-AUC, i.e. it ranks no-shows above "
      "attenders best, and its accuracy beats the baseline. The formal choice is made in 3.5." % best_row)

\ Task 3.4 
RF train acc 1.000 | test acc 0.753 | recall 0.327 | ROC-AUC 0.697
Top 3 importances:
 lead_days      0.223
age            0.202
distance_km    0.190

                   Model  Accuracy  Recall  ROC-AUC
    Logistic regression     0.783   0.306    0.781
Decision tree (depth 3)     0.768   0.286    0.699
          Random forest     0.753   0.327    0.697

On this (reporting-only) table, Logistic regression looks best: highest ROC-AUC, i.e. it ranks no-shows above attenders best, and its accuracy beats the baseline. The formal choice is made in 3.5.


<>:3: SyntaxWarning: invalid escape sequence '\ '
<>:3: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_537/3539498613.py:3: SyntaxWarning: invalid escape sequence '\ '
  print("\ Task 3.4 ")


In [63]:

# ===========================================================================#
# Task 3.5 - Tune, choose model + threshold, test once #
# ===========================================================================#
print("\n=== Task 3.5 ===")
# (a) grid search on training set only
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc").fit(X_train, y_train)
print("(a) best_params_:", grid.best_params_, "| best_score_ (CV ROC-AUC): %.3f" % grid.best_score_)

# (b) choose the model by cross-validated training ROC-AUC
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv = {
    "Logistic regression (pipeline)": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42),
                                               X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (default)": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                               X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned random forest": grid.best_score_,
}
print("\n(b) 5-fold CV ROC-AUC on training set:")
for k, v in cv.items():
    print("   %-32s %.3f" % (k, v))
winner = max(cv, key=cv.get)
print("   Winner:", winner)

if winner.startswith("Logistic"):
    final_model = lr_pipe
elif winner.startswith("Tuned"):
    final_model = RandomForestClassifier(random_state=42, **grid.best_params_)
elif winner.startswith("Random"):
    final_model = RandomForestClassifier(n_estimators=100, random_state=42)
else:
    final_model = DecisionTreeClassifier(max_depth=3, random_state=42)

# (c) choose threshold from out-of-fold training probabilities
oof = cross_val_predict(final_model, X_train, y_train, cv=5, method="predict_proba")[:, 1]
print("\n(c) Out-of-fold training probabilities:")
rows = []
for th in (0.5, 0.35):
    f = (oof >= th).astype(int)
    rows.append({"threshold": th,
                 "precision": precision_score(y_train, f, zero_division=0),
                 "recall": recall_score(y_train, f),
                 "F1": f1_score(y_train, f),
                 "flagged": int(f.sum())})
th_table = pd.DataFrame(rows).round(3)
print(th_table.to_string(index=False))
THRESH = float(th_table.loc[th_table["F1"].idxmax(), "threshold"])
print("Chosen threshold: %.2f (higher F1). Recall matters because a missed warning costs an empty "
      "slot, while an extra call costs only a minute." % THRESH)

# (d) fit winner on training set, evaluate ONCE on test set
final_model.fit(X_train, y_train)
p = final_model.predict_proba(X_test)[:, 1]
flag = (p >= THRESH).astype(int)
print("\n(d) Single test-set evaluation at threshold %.2f:" % THRESH)
print("   precision %.3f | recall %.3f | flagged %d of %d | ROC-AUC %.3f" % (
    precision_score(y_test, flag), recall_score(y_test, flag), flag.sum(), len(flag), roc_auc_score(y_test, p)))

# (e) save test_risk_scores.csv
out = pd.DataFrame({"appointment_id": ids.loc[X_test.index].values,
                    "no_show": y_test.values,
                    "risk_prob": p.round(4),
                    "flag": flag})
out.to_csv("test_risk_scores.csv", index=False)
print("\n(e) Saved test_risk_scores.csv:", out.shape, "| flag = 1 rows:", int(out['flag'].sum()))
print("A real clinic would set the threshold by how many follow-up calls the front desk can make in an "
      "evening, weighed against the cost of an empty slot.")

print("\nLimits (Task 4.4): Task 1.3 medians/quartiles/fences used all 990 rows before the split; the same "
      "patient can sit in both train and test; and the split is random, whereas a real system trains on the "
      "past and predicts the future.")



=== Task 3.5 ===
(a) best_params_: {'max_depth': 3, 'n_estimators': 200} | best_score_ (CV ROC-AUC): 0.765

(b) 5-fold CV ROC-AUC on training set:
   Logistic regression (pipeline)   0.772
   Decision tree (depth 3)          0.680
   Random forest (default)          0.735
   Tuned random forest              0.765
   Winner: Logistic regression (pipeline)

(c) Out-of-fold training probabilities:
 threshold  precision  recall    F1  flagged
      0.50      0.597   0.254 0.357       72
      0.35      0.507   0.456 0.480      152
Chosen threshold: 0.35 (higher F1). Recall matters because a missed warning costs an empty slot, while an extra call costs only a minute.

(d) Single test-set evaluation at threshold 0.35:
   precision 0.551 | recall 0.551 | flagged 49 of 198 | ROC-AUC 0.781

(e) Saved test_risk_scores.csv: (198, 4) | flag = 1 rows: 49
A real clinic would set the threshold by how many follow-up calls the front desk can make in an evening, weighed against the cost of an empty slo